# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


In [2]:
from nba_ou.modeling.modeling import (
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle
)

In [3]:
nan_threshold = 50.0
max_na_per_row = 5

## Load Data

In [4]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260317.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [5]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11249 rows
Basic cleaning complete: 8640 rows remaining

Starting advanced column cleaning with 3240 columns

Advanced column cleaning complete: 3240 → 2137 columns (1103 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 20/136
  Remaining NaN cells: 1030992

Dropping rows with more than 5 NaN values...
Removed 4019 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4621, 2137)


In [6]:
import time
time.sleep(4.2 * 3600)

In [7]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1343,LAST_5_GAMES_TOTAL_POINTS_BEFORE,455,9.85,2021.0
1903,LEAGUE_GAMES_LAST_1D_BEFORE,193,4.18,2024.0
1946,total_consensus_pct_under,67,1.45,2024.0
1342,LAST_4_GAMES_TOTAL_POINTS_BEFORE,65,1.41,2021.0
1947,spread_consensus_pct_home,63,1.36,2024.0
2127,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D_BEFORE,42,0.91,2021.0
2126,TRAVEL_RECENCY_RATIO_HOME_2D_OVER_14D_BEFORE,17,0.37,2024.0
2078,TOP3_AVAILABILITY_EFFECT_AWAY_MAX_ABS_DIFF_FRO...,13,0.28,2022.0
2076,TOP3_AVAILABILITY_EFFECT_AWAY_MAX_ABS_TOTAL_PO...,13,0.28,2022.0
2074,TOP3_AVAILABILITY_EFFECT_AWAY_MEAN_DIFF_FROM_LINE,13,0.28,2022.0


In [8]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [9]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [10]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2021    1143
2022    1154
2023     157
2024    1233
2025     934
dtype: int64


## Train / Test

In [11]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.base import clone

In [12]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 4389
Final test set size: 232
Final test date range: 2026-02-08 00:00:00 -> 2026-03-17 00:00:00


In [13]:
TARGET_COL = "TOTAL_POINTS"

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

X_dev shape: (4389, 2134)
X_test_final shape: (232, 2134)


In [14]:
from nba_ou.modeling.scorers import OverUnderScorerTotalPoints, over_under_betting_accuracy_total_points, evaluate_total_points_thresholds
    
ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
}

def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc == "OU_Betting_Accuracy":
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()

In [15]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=30,
    step_games_between_tests=50,
    train_games=4000,
    min_train_games=2500,
    max_folds=12,
    verbose=1,
)


Created 12 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           3282            30       2021-10-29     2025-03-02      2025-03-03    2025-03-06         2024
    2           3366            36       2021-10-29     2025-03-13      2025-03-14    2025-03-17         2024
    3           3453            36       2021-10-29     2025-03-24      2025-03-25    2025-03-29         2024
    4           3542            33       2021-10-29     2025-04-05      2025-04-06    2025-04-09         2024
    5           3687            32       2021-10-29     2025-06-22      2025-10-27    2025-11-03         2025
    6           3772            30       2021-10-29     2025-11-10      2025-11-11    2025-11-14         2025
    7           3858            30       2021-10-29     2025-11-22      2025-11-23    2025-11-26         2025
    8           3939            31       2021-10-29     2025-12-03      2025

In [16]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MSE: 381.08110
Validation MSE: 385.38276

Train RMSE: 19.52123
Validation RMSE: 19.44557

Train MAE: 15.58984
Validation MAE: 15.66027

Train R2: 0.00000
Validation R2: -0.11195

Train OU_Betting_Accuracy: 49.50%
Validation OU_Betting_Accuracy: 51.78%



In [17]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MSE: 172.35200
Validation MSE: 2512707.83857

Train RMSE: 13.12464
Validation RMSE: 496.08961

Train MAE: 10.18382
Validation MAE: 111.78794

Train R2: 0.54762
Validation R2: -3734.20991

Train OU_Betting_Accuracy: 72.17%
Validation OU_Betting_Accuracy: 46.71%



In [18]:
xgb_reg = XGBRegressor(
    max_depth=3,
    learning_rate=0.054,
    n_estimators=116,
    subsample=0.7,
    colsample_bytree=0.78,
    reg_alpha=7,
    reg_lambda=0.2,
    min_child_weight=13.23,
    gamma=1.82,
    n_jobs=-2,          # choose your CPU count
    random_state=16,
)

cv_results = cross_validate(
    xgb_reg,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,          # leave at 1 because XGB already parallelizes internally
)

print("XGBoost")
print_metrics(cv_results)

XGBoost
Train MSE: 213.52040
Validation MSE: 318.83243

Train RMSE: 14.61055
Validation RMSE: 17.62927

Train MAE: 11.55252
Validation MAE: 14.31718

Train R2: 0.43961
Validation R2: 0.08054

Train OU_Betting_Accuracy: 74.44%
Validation OU_Betting_Accuracy: 50.84%



In [19]:
xgb_reg.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_test_final,
    y_pred_test_total,
    betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 301.09491
RMSE: 17.35209
MAE: 13.53405
OU_Betting_Accuracy: 53.10%


In [20]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,232,100.0%,53.10%
1,1,158,68.1%,52.94%
2,2,99,42.7%,57.89%
3,3,48,20.7%,65.22%
4,4,28,12.1%,66.67%
5,5,12,5.2%,50.00%
6,6,5,2.2%,40.00%
7,7,2,0.9%,50.00%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


# OPTUNA

In [21]:
from nba_ou.modeling.optuna_total_points import (
    tune_xgb_total_points_optuna,
    summarize_optuna_trials,
    fit_best_xgb_total_points,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=4 * 3600,
    objective_name="reg:squarederror",   # second run: reg:pseudohubererror
    study_name="xgb_total_points_mae",
)

print("Best CV MAE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nSecondary metrics from best trial:")
print("Mean RMSE:", study.best_trial.user_attrs.get("mean_rmse"))
print("Mean R2:", study.best_trial.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean best_iteration:", study.best_trial.user_attrs.get("mean_best_iteration"))

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
        }
    )
)

[I 2026-03-18 22:01:48,513] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-18 22:08:28,031] Trial 0 finished with value: 13.801304929931836 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307}. Best is trial 0 with value: 13.801304929931836.
[I 2026-03-18 22:17:26,738] Trial 1 finished with value: 13.805946201622243 and parameters: {'max_depth': 2, 'min_child_weight': 51.819268381786635, 'gamma': 1.7346760026809933, 'subsample': 0.5811969357559948, 'colsample_bytree': 0.675188230006178, 'learning_rate': 0.010426963054371881, 'reg_alpha': 0.06701717253228541, 'reg_lambda': 3.152289132591451}. Best is trial 0 with value: 13.801304929931836.
[I 2026-03-18 22:22:32,229] Trial 2 finished with value: 13.775607746014073 and parameters: {'max_depth': 4, 'min_child_weight': 15.848753157115912, 'gamma': 0.7236802167715846, 'subsample':

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda
0,48,13.5658,17.1192,0.1375,58.61%,138,3,10.710333,0.243804,0.570254,0.391372,0.047356,0.044299,8.792696
1,14,13.5940,17.0613,0.1421,61.54%,83,3,19.841733,2.296581,0.674943,0.594567,0.042276,0.204752,1.041225
2,49,13.5988,17.1818,0.1314,56.99%,104,4,13.451327,0.296585,0.570472,0.384122,0.037726,0.017579,19.471256
3,31,13.6018,17.1405,0.1351,59.65%,99,3,17.456382,2.861530,0.638251,0.524037,0.044023,1.215392,1.831228
4,28,13.6184,17.0676,0.1421,57.61%,158,3,9.655201,2.778160,0.592960,0.433234,0.033966,1.432730,1.510608
5,45,13.6237,17.2470,0.1246,57.25%,120,3,10.764096,0.773820,0.572457,0.503123,0.046958,0.183676,3.623879
6,21,13.6259,17.1029,0.1402,56.21%,79,3,11.256366,2.893255,0.661435,0.469068,0.058626,0.208338,3.957160
7,15,13.6305,17.1195,0.1365,58.41%,111,3,11.940574,2.828411,0.648410,0.578888,0.042969,0.332094,2.411632
8,22,13.6566,17.1216,0.1363,55.79%,95,3,20.581853,2.976486,0.646590,0.530682,0.058244,0.118123,2.726036
9,23,13.6569,17.1546,0.1331,58.76%,92,3,11.768267,2.608323,0.698232,0.353045,0.044724,0.253481,1.605096


In [35]:
df_dev


,TOTAL_LINE_bet365,GAME_NUMBER_TEAM_AWAY,GAME_DATE,IS_PLAYOFF_GAME_BEFORE,PLAYOFF_GAMES_LAST_SEASON_TEAM_AWAY,PLAYOFF_GAMES_LAST_SEASON_TEAM_HOME,SEASON_YEAR,WINS_BEFORE_THIS_GAME_TEAM_HOME,TEAM_RECORD_BEFORE_GAME_TEAM_HOME,REST_DAYS_BEFORE_MATCH_TEAM_HOME,...,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D_BEFORE,REST_DAYS_DIFF_HOME_MINUS_AWAY_BEFORE,INJURY_PTS_SHARE_HOME_BEFORE,INJURY_PTS_SHARE_AWAY_BEFORE,STAR_PTS_PCT_DIFF_HOME_MINUS_AWAY_BEFORE,POSS_X_TSPCT_HOME_BEFORE,POSS_X_TSPCT_AWAY_BEFORE,IS_WEEKEND_BEFORE,MONTH_BEFORE,IS_US_HOLIDAY_BEFORE
0,223.0,6,2021-10-29,0,0.0,12.0,2021,2,0.400000,2,...,0.836507,0,0.112694,0.246422,0.063897,52.599556,58.291120,0,10,False
1,222.0,6,2021-10-29,0,0.0,4.0,2021,3,0.750000,2,...,0.806559,0,0.040984,0.000000,-0.024599,60.486000,62.040333,0,10,False
2,222.0,5,2021-10-29,0,0.0,0.0,2021,1,0.200000,2,...,0.882259,0,0.008481,0.000000,-0.031300,49.033500,58.349500,0,10,False
3,214.0,6,2021-10-30,0,5.0,0.0,2021,1,0.166667,1,...,0.845034,-1,0.255414,0.000000,-0.082475,49.527222,57.024000,1,10,False
4,216.5,6,2021-10-30,0,18.0,12.0,2021,3,0.600000,2,...,0.885365,0,0.000000,0.000000,-0.000039,58.996000,49.875111,1,10,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4384,217.0,51,2026-02-07,0,0.0,0.0,2025,13,0.260000,2,...,0.867881,0,0.044101,0.480622,0.113531,53.976124,56.218263,1,2,False
4385,234.5,53,2026-02-07,0,0.0,5.0,2025,26,0.520000,2,...,0.786074,0,0.197700,0.182497,-0.029389,59.263520,57.961244,1,2,False
4386,233.5,52,2026-02-07,0,0.0,0.0,2025,35,0.686275,2,...,0.000000,0,0.078862,0.132285,0.004072,60.254424,57.147103,1,2,False
4387,213.5,51,2026-02-07,0,7.0,23.0,2025,40,0.769231,3,...,0.000000,1,0.561009,0.227228,-0.081578,60.018238,54.074417,1,2,False


In [39]:
total_df = df_dev.tail(3500)

In [40]:
X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")

In [41]:
best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    study=study,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 291.44849
RMSE: 17.07186
MAE: 13.36117
OU_Betting_Accuracy: 54.87%


In [43]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics
df_to_train_split_rows = df_to_train.copy()  # if you want to keep the original df_to_train unchanged
df_to_train_split_rows = df_to_train_split_rows.tail(4000)  # keep only the last 4000 rows for training the final model

X_full = df_to_train_split_rows.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train_split_rows[TARGET_COL], errors="coerce")

production_model = fit_best_xgb_total_points(
    X_dev=X_full,
    y_dev=y_full,
    study=study,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train_split_rows["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"five_seasons_xgb_total_points_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="five_seasons_total_points",
        prediction_source="five_seasons_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=study.best_trial.params,
        mean_best_iteration=study.best_trial.user_attrs.get("mean_best_iteration"),
        cv_mae=float(study.best_value),
        cv_rmse=study.best_trial.user_attrs.get("mean_rmse"),
        cv_ou_acc=study.best_trial.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train_split_rows["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows["GAME_DATE"].max().to_pydatetime(),
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/5_seasons/",
    metadata=metadata,
)

print(f"Production model trained on {len(X_full)} rows using fixed n_estimators from mean_best_iteration.")
print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Production model trained on 4000 rows using fixed n_estimators from mean_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/5_seasons/five_seasons_xgb_total_points_17_03_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/5_seasons/five_seasons_xgb_total_points_17_03_26.meta.json


In [44]:
study.best_trial.user_attrs.get("mean_best_iteration")

138